# Vues mensuelles par contribution — générique vs. personnalisées

Pour **chaque contribution** de code.travail.gouv.fr, le nombre de **vues par mois**
de la page **générique** (`/contribution/{slug}`) comparé à la **somme** des pages
**personnalisées** par convention collective (`/contribution/{idcc}-{slug}` et
`/contribution/{slug}/{idcc}`), sur **janvier → juin 2026**, avec le **cumul** sur
la période.

La liste des contributions est récupérée depuis le
[sitemap](https://code.travail.gouv.fr/sitemap.xml) ; les vues viennent de l'**API
Reporting Matomo** (`MATOMO_*` dans `.env`).

> Astuce : mettre `DEMO = True` pour un aperçu instantané sur données synthétiques,
> sans credentials ni réseau.

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import pandas as pd
from analysis.connectors.matomo import MatomoSQLConnector

matomo = MatomoSQLConnector()
await matomo.connect()

In [ ]:
# Depuis le début de l'année
interval_start = '2026-01-01 00:00:00'
interval_stop = '2026-07-01 00:00:00'

## 1. Récupération des données

On récupère les données depuis la BDD

In [ ]:
columns = ['action_id', 'idvisit', 'action_type', 'action_url', 'action_timestamp']

# Trois cas couverts :
#  - generique :      contribution/les-conges-pour-evenements-familiaux
#  - ancien format :  contribution/9999-les-conges-pour-evenements-familiaux
#  - nouveau format : contribution/les-conges-pour-evenements-familiaux/9999-nom-de-la-cc
url_pattern = (
    r'code\.travail\.gouv\.fr/contribution/'
    r'('
    r'([0-9]{2,4}-)?les-conges-pour-evenements-familiaux($|[?#])'      # generique + ancien format
    r'|les-conges-pour-evenements-familiaux/[0-9]{2,4}-'               # nouveau format
    r')'
)

query = f"""
    SELECT {", ".join(columns)}
    FROM matomo_partitioned
    WHERE action_timestamp >= '{interval_start}'
      AND action_timestamp < '{interval_stop}'
      AND action_type = 'action'
      AND action_url ~ '{url_pattern}'
    ORDER BY action_timestamp ASC;
"""

In [ ]:
visits_data = await matomo.run_query(query)

In [ ]:
visits_df = pd.DataFrame(visits_data, columns=columns)

## Analyse des données

On aggrège par semaine et par idcc.

In [ ]:
visits_df['action_timestamp'] = pd.to_datetime(visits_df['action_timestamp'])
visits_df['semaine'] = visits_df['action_timestamp'].dt.to_period('W-SUN').dt.start_time.dt.date

url = visits_df['action_url']

# Nouveau format (prioritaire) : /contribution/les-conges-.../9999-nom-de-la-cc
idcc_nouveau = url.str.extract(
    r'/contribution/les-conges-pour-evenements-familiaux/([0-9]{2,4})-'
)[0]

# Ancien format : /contribution/9999-les-conges-...
# On l'applique uniquement aux lignes non déjà classées en nouveau format
idcc_ancien = url.str.extract(
    r'/contribution/([0-9]{2,4})-les-conges-pour-evenements-familiaux'
)[0]
idcc_ancien = idcc_ancien.where(idcc_nouveau.isna())

visits_df['idcc'] = idcc_nouveau.combine_first(idcc_ancien)

# Générique : le slug SANS rien derrière (ni /, donc pas les pages filles)
est_generique = url.str.contains(
    r'/contribution/les-conges-pour-evenements-familiaux($|[?#])'
) & visits_df['idcc'].isna()

visits_df.loc[est_generique, 'idcc'] = 'generique'

# Tout le reste est marqué explicitement pour vérification, plus de fillna silencieux
visits_df['idcc'] = visits_df['idcc'].fillna('a_verifier')

In [ ]:
tableau = (
    visits_df
    .groupby(['semaine', 'idcc'])['idvisit']
    .nunique()
    .unstack('idcc', fill_value=0)
    .sort_index()
)

cols = ['generique'] + [c for c in tableau.columns if c != 'generique']
tableau = tableau[cols]

## Résultat

Graphique représentant les vues par contribution, par semaine.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

for col in tableau.columns:
    fig.add_trace(go.Scatter(
        x=tableau.index,
        y=tableau[col],
        mode='lines+markers',
        name=str(col),
        # La générique visible par défaut, les CC masquées (cliquables dans la légende)
        visible=True if col == 'generique' else 'legendonly',
    ))

fig.update_layout(
    title="Visites uniques par semaine — les congés pour événements familiaux",
    xaxis_title="Semaine",
    yaxis_title="Nombre de visites uniques",
    hovermode='x unified',
    legend_title="Page (générique / IDCC)",
    height=600,
)

# Repère visuel du changement d'URL du 24 mars
fig.add_vline(x='2026-03-23', line_dash='dash', line_color='grey',
              annotation_text='Migration URLs (24/03)', annotation_position='top')

fig.show()

Graphique avec les contributions sommées par semaine.

In [ ]:
# Colonnes CC = toutes sauf la générique
cols_cc = [c for c in tableau.columns if c != 'generique']

tableau_agrege = pd.DataFrame({
    'generique': tableau['generique'],
    'toutes_cc': tableau[cols_cc].sum(axis=1),
})

fig = go.Figure()

for col in tableau_agrege.columns:
    fig.add_trace(go.Scatter(
        x=tableau_agrege.index,
        y=tableau_agrege[col],
        mode='lines+markers',
        name=col,
    ))

fig.update_layout(
    title="Visites uniques par semaine — générique vs pages CC (agrégées)",
    xaxis_title="Semaine",
    yaxis_title="Nombre de visites uniques",
    hovermode='x unified',
    height=500,
)

fig.add_vline(x='2026-03-23', line_dash='dash', line_color='grey',
              annotation_text='Migration URLs (24/03)', annotation_position='top')

fig.show()